# Cross-Modal Diagnostic Observability — Stage 11E-R

## tl;dr

This development-only notebook applies the frozen Stage 11D-R grouped partitions to four breast-ultrasound source domains, extracts one fixed ImageNet representation, measures source recoverability with grouped OOF image-level AUC and separately observed held-out image-level AUC with released-group cluster bootstrap, freezes every source axis before held-out validation, and authorises outgoing development edges only from sources that pass the pre-specified gate.

## Context & Methods

Stage 11D-R retained 3,306 binary lesion images and authorised source recoverability for four split-ready development domains. Stage 11E-R is not a transfer or DDO stage. It determines whether each source can express its own malignant-versus-benign endpoint under one fixed representation and one fixed linear probe.

### Key assumptions and hard boundaries

- Representation: torchvision ResNet-50 with `IMAGENET1K_V2` weights, canonical weight transforms, global-average-pooled 2,048-D features, and per-image L2 normalisation.
- Probe: `StandardScaler` plus class-balanced L2 logistic regression with fixed `C=1.0`; images are weighted so each released patient/lesion group contributes total training weight one.
- Development estimate: frozen grouped OOF folds from Stage 11D-R; no fold, model, regularisation, preprocessing, threshold, or feature selection is tuned.
- Validation estimate: the separately frozen grouped held-out partition is scored only after the final development axis and the pre-validation freeze record exist.
- Gate: development OOF AUC and held-out AUC must each be at least 0.70, and both group-bootstrap 95% CI lower bounds must be strictly greater than 0.55.
- AUC is computed at the released binary image/lesion endpoint level; uncertainty resamples whole released patient/lesion groups as clusters. Mixed-endpoint groups remain intact within one partition and one OOF fold.
- A failed source is retired for this protocol and is not rescued. Its frozen audit axis remains an unauthorised artefact.
- Stage 12, DDO-2 fitting, transfer-outcome calibration, and every locked-blind asset remain prohibited.


In [1]:
# @title 11E-R-0. Mount Drive, verify the sealed Stage 11D-R handoff, and freeze the protocol
import gc, hashlib, io, json, os, platform, random, re, sys, zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps

try:
    from IPython.display import display
except Exception:
    display = print

IN_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except Exception:
    pass

DEFAULT_ROOT = Path('/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability') if IN_COLAB else Path('/tmp/Cross-Modal_Diagnostic_Observability')
PROJECT_ROOT = Path(os.environ.get('CDO_PROJECT_ROOT', str(DEFAULT_ROOT)))
CODE_ROOT = PROJECT_ROOT/'05_Code'/'Cross_Modal'
CM_ROOT = PROJECT_ROOT/'06_Data_Records'/'Cross_Modal'
PARENT_ROOT = CM_ROOT/'Stage11D-R_Cross_Roster_Dedup_Exact_Manifest_And_Grouped_Split_Freeze_v0.1'
ROOT = CM_ROOT/'Stage11E-R_Development_Only_Source_Recoverability_And_Axis_Freeze_v0.1'
RECEIPT_ROOT = PROJECT_ROOT/'00_Data_Acquisition'/'Stage11C_Manual_Official_Receipts'
P0,P1,P2,P3,P4,P5,P6 = [ROOT/x for x in [
    '00_Protocol','01_Frozen_Embeddings','02_Development_OOF','03_Frozen_Source_Axes',
    '04_Heldout_Validation','05_Firewall','06_Results'
]]
for p in [CODE_ROOT,P0,P1,P2,P3,P4,P5,P6]:
    p.mkdir(parents=True, exist_ok=True)

NOTEBOOK_NAME = 'CrossModal_Stage11E-R_Development_Only_Source_Recoverability_And_Axis_Freeze_v0.1.ipynb'
NOTEBOOK_PATH = CODE_ROOT/NOTEBOOK_NAME
PARENT_PROTOCOL = PARENT_ROOT/'00_Protocol'/'Stage11D-R_Protocol_Seal_v0.1.json'
PARENT_EXACT = PARENT_ROOT/'01_Exact_Manifest'/'Stage11D-R_Frozen_Exact_Image_Label_Group_Manifest_v0.1.csv'
PARENT_DEDUP = PARENT_ROOT/'02_Deduplication'/'Stage11D-R_Deduplication_And_Endpoint_Summary_v0.1.csv'
PARENT_GROUPS = PARENT_ROOT/'03_Grouped_Splits'/'Stage11D-R_Frozen_Group_Assignments_v0.1.csv'
PARENT_SPLITS = PARENT_ROOT/'03_Grouped_Splits'/'Stage11D-R_Frozen_Grouped_Split_Manifest_v0.1.csv'
PARENT_HANDOFF = PARENT_ROOT/'03_Grouped_Splits'/'Stage11D-R_Stage11E-R_Handoff_v0.1.json'
PARENT_FINAL = PARENT_ROOT/'05_Results'/'Stage11D-R_Complete_v0.1.json'

EXPECTED_PARENT_PROTOCOL = '8f2c409cf51429c33aea93b0c8788484c53c909865c4153ca88de3bb3877cc98'
EXPECTED_PARENT_EXACT = '3e66aad296106d12958a8d84634e5cc780399329ba22b503e49f42019bf3db11'
EXPECTED_PARENT_DEDUP = '6e41e091759cfea56bbfb42ec841677d3e0b844a86ab882e486251ec3886df13'
EXPECTED_RETAINED_IMAGES = 3306
EXPECTED_SPLIT_READY_DOMAINS = 4

FOLDERS = {
    'BUS_BRA_2024':'BUS_BRA',
    'BUSI_WHU_2025_V3':'BUSI_WHU',
    'BREAST_LESIONS_USG_2024':'BREAST_LESIONS_USG',
    'BUS_UCLM_2025_V3':'BUS_UCLM',
    'RODRIGUES_BUI_2017':'RODRIGUES_BUI',
}
LOCKED = ['BUSI_CAIRO_2019','OASBUD_2017','DERM7PT_2019']
RANDOM_SEED = 20260721
N_BOOTSTRAP = 2000
AUC_THRESHOLD = 0.70
CI_LOWER_THRESHOLD = 0.55
LOGISTIC_C = 1.0
LOGISTIC_MAX_ITER = 10000
MINIMUM_RECOVERABLE_SOURCES = 3
BATCH_SIZE_GPU = 64
BATCH_SIZE_CPU = 16
NUM_WORKERS = 0
AXIS_EQUIVALENCE_TOLERANCE = 1e-8
EXPECTED_MIXED_ENDPOINT_GROUP_IDS = (
    'BUS_UCLM_2025_V3::CHVI', 'BUS_UCLM_2025_V3::FLBA',
    'BUS_UCLM_2025_V3::FUHI', 'BUS_UCLM_2025_V3::HUBL',
    'BUS_UCLM_2025_V3::MENE',
)
LEGACY_ABORTED_PROTOCOL_SEAL = '32bce038793fcba03a960d87e278fcf57d1c6fca1d51dbe3be731c37a3e23b89'

PROTOCOL = P0/'Stage11E-R_Protocol_Seal_v0.1.json'
ABORTED_PROTOCOL = P0/'Stage11E-R_Protocol_Seal_v0.1_ABORTED_PREEXECUTION_MIXED_ENDPOINT_ASSUMPTION.json'
PARENT_COMMIT = P0/'Stage11E-R_Parent_Input_Commitment_v0.1.csv'
ENVIRONMENT = P0/'Stage11E-R_Execution_Environment_v0.1.json'
SOURCE_INVENTORY = P1/'Stage11E-R_Source_Representation_Inventory_v0.1.csv'
GROUP_ENDPOINT_AUDIT = P1/'Stage11E-R_Group_Endpoint_Composition_Audit_v0.1.csv'
EMBEDDING_MANIFEST = P1/'Stage11E-R_Frozen_Embedding_Manifest_v0.1.csv'
OOF_PREDICTIONS = P2/'Stage11E-R_Development_OOF_Predictions_v0.1.csv'
OOF_SUMMARY = P2/'Stage11E-R_Development_OOF_Metric_Summary_v0.1.csv'
AXIS_MANIFEST = P3/'Stage11E-R_Frozen_Source_Axis_Manifest_v0.1.csv'
AXIS_FREEZE = P3/'Stage11E-R_Prevalidation_Axis_Freeze_v0.1.json'
HELDOUT_PREDICTIONS = P4/'Stage11E-R_Heldout_Predictions_v0.1.csv'
HELDOUT_SUMMARY = P4/'Stage11E-R_Heldout_Metric_Summary_v0.1.csv'
RECOVERABILITY_SUMMARY = P4/'Stage11E-R_Source_Recoverability_Decisions_v0.1.csv'
EDGE_ROSTER = P4/'Stage11E-R_Authorised_Development_Edge_Roster_v0.1.csv'
HANDOFF = P4/'Stage11E-R_Stage11F-R_Handoff_v0.1.json'
FIREWALL = P5/'Stage11E-R_Independent_Validity_And_Firewall_Checks_v0.1.csv'
REPORT = P6/'Stage11E-R_Source_Recoverability_And_Axis_Freeze_Report_v0.1.md'
OUTPUT_MANIFEST = P6/'Stage11E-R_Output_Integrity_Manifest_v0.1.csv'
FINAL = P6/'Stage11E-R_Complete_v0.1.json'
RUNTIME = P6/'Stage11E-R_Runtime_State_v0.1.json'

def now():
    return datetime.now(timezone.utc).isoformat()

def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024*1024), b''):
            h.update(block)
    return h.hexdigest()

def sha_bytes(data):
    return hashlib.sha256(data).hexdigest()

def sha_json(value):
    payload = json.dumps(value, sort_keys=True, separators=(',',':'), ensure_ascii=False).encode()
    return hashlib.sha256(payload).hexdigest()

def canonical_csv(frame):
    return frame.fillna('').to_csv(index=False, lineterminator='\n', float_format='%.12g')

def write_text_once(path, text):
    path = Path(path)
    if path.exists():
        assert path.read_text(encoding='utf-8') == text, f'Replay mismatch: {path}'
    else:
        path.write_text(text, encoding='utf-8')

def write_csv_once(path, frame):
    write_text_once(path, canonical_csv(frame))

def write_json_once(path, value):
    write_text_once(path, json.dumps(value, indent=2, ensure_ascii=False)+'\n')

def verify_self(path, hash_field, expected=None):
    value = json.loads(Path(path).read_text(encoding='utf-8'))
    claimed = value[hash_field]
    payload = dict(value)
    payload.pop(hash_field)
    assert sha_json(payload) == claimed, f'Self-hash mismatch: {path}'
    if expected is not None:
        assert claimed == expected, f'Unexpected sealed hash: {path}'
    return value

def create_or_verify_seal(path, payload, hash_field, time_field):
    path = Path(path)
    if path.exists():
        value = verify_self(path, hash_field)
        for key, expected in payload.items():
            assert value[key] == expected, f'Sealed field changed: {path} :: {key}'
        return value
    value = dict(payload)
    value[time_field] = now()
    value[hash_field] = sha_json(value)
    write_json_once(path, value)
    return value

def markdown_table(frame):
    x = frame.fillna('').astype(str)
    esc = lambda s: str(s).replace('|','\\|').replace('\n',' ')
    return '\n'.join(
        ['| '+' | '.join(esc(c) for c in x.columns)+' |',
         '| '+' | '.join('---' for _ in x.columns)+' |'] +
        ['| '+' | '.join(esc(v) for v in row)+' |' for row in x.itertuples(index=False, name=None)]
    )

required = [NOTEBOOK_PATH, PARENT_PROTOCOL, PARENT_EXACT, PARENT_DEDUP, PARENT_GROUPS, PARENT_SPLITS, PARENT_HANDOFF, PARENT_FINAL]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, 'Missing sealed Stage 11D-R inputs:\n'+'\n'.join(missing)

assert sha_file(PARENT_PROTOCOL) != '', 'Parent protocol unreadable'
parent_protocol = verify_self(PARENT_PROTOCOL, 'seal_sha256', EXPECTED_PARENT_PROTOCOL)
parent_handoff = verify_self(PARENT_HANDOFF, 'handoff_sha256')
parent_final = verify_self(PARENT_FINAL, 'final_record_sha256')
assert sha_file(PARENT_EXACT) == EXPECTED_PARENT_EXACT
assert sha_file(PARENT_DEDUP) == EXPECTED_PARENT_DEDUP
assert sha_file(PARENT_EXACT) == parent_final['exact_manifest_sha256'] == parent_handoff['exact_manifest_sha256']
assert sha_file(PARENT_DEDUP) == parent_final['dedup_summary_sha256'] == parent_handoff['dedup_summary_sha256']
assert sha_file(PARENT_GROUPS) == parent_final['group_assignment_sha256'] == parent_handoff['group_assignment_sha256']
assert sha_file(PARENT_SPLITS) == parent_final['split_manifest_sha256'] == parent_handoff['split_manifest_sha256']
assert parent_final['retained_binary_images'] == EXPECTED_RETAINED_IMAGES
assert parent_final['source_recoverability_authorised'] is True
assert parent_handoff['source_recoverability_authorised'] is True
assert parent_handoff['split_ready_domain_count'] == EXPECTED_SPLIT_READY_DOMAINS
assert parent_handoff['minimum_recoverable_sources_for_global_edge_gate'] == MINIMUM_RECOVERABLE_SOURCES
assert parent_final['stage12_authorised'] is False and parent_final['ddo2_fitted'] is False
assert parent_handoff['locked_blind_assets_touched'] is False
assert ABORTED_PROTOCOL.is_file(), 'Missing preserved aborted pre-execution protocol seal'
aborted_protocol = verify_self(ABORTED_PROTOCOL, 'seal_sha256', LEGACY_ABORTED_PROTOCOL_SEAL)
assert aborted_protocol['heldout_performance_observed_before_protocol_freeze'] is False

SOURCE_DATASETS = list(parent_handoff['split_ready_dataset_ids'])
assert len(SOURCE_DATASETS) == EXPECTED_SPLIT_READY_DOMAINS
assert set(SOURCE_DATASETS).issubset(FOLDERS)

commit = pd.DataFrame([
    {'role':path.name, 'relative_path':str(path.relative_to(PROJECT_ROOT)), 'size_bytes':path.stat().st_size, 'sha256':sha_file(path)}
    for path in required[1:]
])
write_csv_once(PARENT_COMMIT, commit)

analysis_spec = {
    'scope':'FOUR_SPLIT_READY_DEVELOPMENT_DOMAINS_SOURCE_RECOVERABILITY_ONLY',
    'representation':{
        'library':'torchvision',
        'architecture':'resnet50',
        'weights':'ResNet50_Weights.IMAGENET1K_V2',
        'transform':'weight-enum canonical resize/crop/RGB/ImageNet normalisation',
        'feature':'2048-dimensional global-average-pooled penultimate activation',
        'normalisation':'per-image L2'
    },
    'probe':{
        'scaler':'StandardScaler default fixed settings',
        'classifier':'class-balanced L2 LogisticRegression',
        'C':LOGISTIC_C,
        'solver':'liblinear',
        'max_iter':LOGISTIC_MAX_ITER,
        'training_weight':'each released group contributes total weight one'
    },
    'metric':{
        'endpoint_unit':'released binary image/lesion endpoint',
        'point_estimate':'image-level ROC AUC',
        'uncertainty':'released-group cluster bootstrap preserving every sampled group and all endpoint labels within it',
        'mixed_endpoint_groups':'permitted and kept intact within one partition and one OOF fold',
        'bootstrap_replicates':N_BOOTSTRAP,
        'gate':'development OOF and held-out image-level AUC each >=0.70; both released-group cluster-bootstrap 95% CI lower bounds >0.55'
    },
    'frozen_seed':RANDOM_SEED,
    'minimum_recoverable_sources':MINIMUM_RECOVERABLE_SOURCES,
    'prohibited':['model or C tuning','class-weight tuning','threshold selection','preprocessing selection','feature selection','failed-source rescue','transfer outcome evaluation','DDO2','Stage12','locked-blind assets']
}
protocol_payload = {
    'stage':'Stage11E-R',
    'version':'0.1',
    'parent_stage11d_r_final_sha256':parent_final['final_record_sha256'],
    'parent_stage11d_r_handoff_sha256':parent_handoff['handoff_sha256'],
    'parent_exact_manifest_sha256':sha_file(PARENT_EXACT),
    'parent_group_assignment_sha256':sha_file(PARENT_GROUPS),
    'parent_split_manifest_sha256':sha_file(PARENT_SPLITS),
    'source_dataset_ids':SOURCE_DATASETS,
    'analysis_spec':analysis_spec,
    'pre_execution_protocol_correction':{
        'reason':'SEALED_STAGE11D_R_CONTAINS_VALID_MIXED_ENDPOINT_GROUPS',
        'legacy_aborted_protocol_seal_sha256':LEGACY_ABORTED_PROTOCOL_SEAL,
        'expected_mixed_endpoint_group_ids':list(EXPECTED_MIXED_ENDPOINT_GROUP_IDS),
        'resolution':'IMAGE_LEVEL_AUC_WITH_RELEASED_GROUP_CLUSTER_BOOTSTRAP',
        'heldout_performance_observed_before_correction':False,
    },
    'heldout_performance_observed_before_protocol_freeze':False,
    'stage12_authorised':False,
    'ddo2_fitted':False,
    'locked_blind_assets_touched':False,
}
protocol = create_or_verify_seal(PROTOCOL, protocol_payload, 'seal_sha256', 'sealed_utc')
REPLAY = FINAL.is_file()
runtime = {
    'stage':'Stage11E-R',
    'replay_mode':REPLAY,
    'source_dataset_ids':SOURCE_DATASETS,
    'heldout_validation_observed':False,
    'transfer_outcomes_evaluated':False,
    'ddo2_fitted':False,
    'stage12_authorised':False,
    'locked_blind_assets_touched':False,
}
print('Stage 11D-R final / handoff verified:', parent_final['final_record_sha256'], parent_handoff['handoff_sha256'])
print('Stage 11E-R protocol seal / replay:', protocol['seal_sha256'], REPLAY)
print('Authorised source domains:', SOURCE_DATASETS)
print('Pre-execution metric correction: image-level AUC with released-group cluster bootstrap; legacy seal preserved:', LEGACY_ABORTED_PROTOCOL_SEAL)


Mounted at /content/drive
Stage 11D-R final / handoff verified: 0e35ab95ad1c087efa43de704bdce11152745b812efb438bba92de390cf047a7 e68d4e1ffccb9554c549c29a7af3d490c4a9fbc62e7bf082b3bdb5b6c36d332f
Stage 11E-R protocol seal / replay: 4362d6a8baed676ee6245971dac40aeb8d2865541df05446ce970ed4431fba72 False
Authorised source domains: ['BUS_BRA_2024', 'BUSI_WHU_2025_V3', 'BUS_UCLM_2025_V3', 'RODRIGUES_BUI_2017']
Pre-execution metric correction: image-level AUC with released-group cluster bootstrap; legacy seal preserved: 32bce038793fcba03a960d87e278fcf57d1c6fca1d51dbe3be731c37a3e23b89


In [2]:
# @title 11E-R-1. Reconstruct the sealed virtual-image index and audit grouped partitions
IMAGE_EXT = {'.png','.jpg','.jpeg','.bmp','.tif','.tiff'}
zip_handles, zip_buffers, entry_lookup = [], [], {}

def sealed_virtual_path(parts):
    # Stage 11D-R writes nested archive boundaries as `!/`; preserve that exact key syntax.
    return '!/'.join(parts)

assert sealed_virtual_path(('outer/archive.zip','img/00001.bmp')) == 'outer/archive.zip!/img/00001.bmp'

def register_zip(dataset_id, outer_path, archive, prefix=(), depth=0):
    zip_handles.append(archive)
    for info in archive.infolist():
        if info.is_dir():
            continue
        chain = tuple(prefix)+(info.filename,)
        virtual_path = sealed_virtual_path(chain)
        suffix = Path(info.filename).suffix.lower()
        entry_lookup[(dataset_id, virtual_path)] = {
            'zip':archive, 'member':info.filename, 'suffix':suffix,
            'size':int(info.file_size), 'outer':outer_path.name, 'depth':depth
        }
        if suffix == '.zip':
            nested_bytes = archive.read(info)
            buffer = io.BytesIO(nested_bytes)
            zip_buffers.append(buffer)
            register_zip(dataset_id, outer_path, zipfile.ZipFile(buffer), chain, depth+1)

def read_virtual(dataset_id, virtual_path):
    entry = entry_lookup[(dataset_id, virtual_path)]
    return entry['zip'].read(entry['member'])

for dataset_id in SOURCE_DATASETS:
    folder = RECEIPT_ROOT/FOLDERS[dataset_id]
    assert folder.is_dir(), f'Missing official receipt folder: {folder}'
    receipt_files = sorted(path for path in folder.iterdir() if path.is_file() and not path.name.startswith('.'))
    assert receipt_files, f'No official receipt files: {folder}'
    for path in receipt_files:
        if path.suffix.lower() == '.zip':
            register_zip(dataset_id, path, zipfile.ZipFile(path))

split_manifest = pd.read_csv(PARENT_SPLITS)
exact_manifest = pd.read_csv(PARENT_EXACT)
group_assignment = pd.read_csv(PARENT_GROUPS)
required_columns = {
    'dataset_id','sample_id','image_virtual_path','raw_image_sha256','binary_label',
    'group_id','partition','oof_fold','split_seed'
}
assert required_columns.issubset(split_manifest.columns), sorted(required_columns-set(split_manifest.columns))
assert len(split_manifest) == EXPECTED_RETAINED_IMAGES == len(exact_manifest)
assert split_manifest.sample_id.is_unique
assert set(split_manifest.dataset_id.unique()) == set(SOURCE_DATASETS)
assert set(split_manifest.partition.unique()) == {'development','heldout'}
assert set(split_manifest.binary_label.astype(int).unique()) == {0,1}
# A released group is a leakage/cluster unit, not necessarily a single-label endpoint unit.
assert not split_manifest.image_virtual_path.astype(str).str.contains(r'(?:^|[/!])(?:masks?|gt)(?:[/!]|$)|_tumou?r\.', case=False, regex=True).any()
inventory_text = '|'.join(split_manifest.image_virtual_path.astype(str)).lower()
assert not any(token.lower() in inventory_text for token in LOCKED)

missing_virtual = [
    (row.dataset_id, row.image_virtual_path)
    for row in split_manifest.itertuples()
    if (row.dataset_id, row.image_virtual_path) not in entry_lookup
]
assert not missing_virtual, f'Missing virtual source images: {missing_virtual[:10]}'
assert split_manifest.groupby(['dataset_id','group_id']).partition.nunique().max() == 1
development_groups = group_assignment[group_assignment.partition=='development']
assert development_groups.groupby(['dataset_id','group_id']).oof_fold.nunique().max() == 1
assert (group_assignment.loc[group_assignment.partition=='heldout','oof_fold'].astype(int) == -1).all()

group_endpoint_audit = (
    split_manifest.groupby(['dataset_id','group_id'], sort=True, as_index=False)
    .agg(
        partition=('partition','first'), oof_fold=('oof_fold','first'),
        images=('sample_id','size'), endpoint_classes=('binary_label','nunique'),
        negative_images=('binary_label', lambda values: int((values.astype(int)==0).sum())),
        positive_images=('binary_label', lambda values: int((values.astype(int)==1).sum())),
    )
)
group_endpoint_audit['mixed_endpoint_group'] = group_endpoint_audit.endpoint_classes.astype(int) > 1
mixed_endpoint_group_ids = group_endpoint_audit.loc[group_endpoint_audit.mixed_endpoint_group, 'group_id'].astype(str).tolist()
assert set(mixed_endpoint_group_ids) == set(EXPECTED_MIXED_ENDPOINT_GROUP_IDS), (
    'Unexpected mixed-endpoint group composition', sorted(mixed_endpoint_group_ids)
)
assert set(group_endpoint_audit.loc[group_endpoint_audit.mixed_endpoint_group,'dataset_id']) == {'BUS_UCLM_2025_V3'}
write_csv_once(GROUP_ENDPOINT_AUDIT, group_endpoint_audit)

inventory_rows = []
for dataset_id in SOURCE_DATASETS:
    frame = split_manifest[split_manifest.dataset_id==dataset_id]
    for partition, part in frame.groupby('partition', sort=True):
        inventory_rows.append({
            'dataset_id':dataset_id,
            'partition':partition,
            'images':len(part),
            'groups':part.group_id.nunique(),
            'negative_images':int((part.binary_label.astype(int)==0).sum()),
            'positive_images':int((part.binary_label.astype(int)==1).sum()),
            'mixed_endpoint_groups':int((part.groupby('group_id').binary_label.nunique()>1).sum()),
            'oof_folds':int(frame.loc[frame.partition=='development','oof_fold'].nunique()),
        })
source_inventory = pd.DataFrame(inventory_rows)
write_csv_once(SOURCE_INVENTORY, source_inventory)
display(source_inventory)
print('Frozen source images indexed:', len(split_manifest))
print('Audited mixed-endpoint groups kept intact:', mixed_endpoint_group_ids)
print('No held-out performance has been computed or displayed.')


,dataset_id,partition,images,groups,negative_images,positive_images,mixed_endpoint_groups,oof_folds
0,BUS_BRA_2024,development,1498,852,1013,485,0,5
1,BUS_BRA_2024,heldout,377,212,255,122,0,5
2,BUSI_WHU_2025_V3,development,740,652,447,293,0,5
3,BUSI_WHU_2025_V3,heldout,186,163,112,74,0,5
4,BUS_UCLM_2025_V3,development,198,29,131,67,4,5
5,BUS_UCLM_2025_V3,heldout,65,7,43,22,1,5
6,RODRIGUES_BUI_2017,development,194,194,77,117,0,5
7,RODRIGUES_BUI_2017,heldout,48,48,19,29,0,5


Frozen source images indexed: 3306
Audited mixed-endpoint groups kept intact: ['BUS_UCLM_2025_V3::CHVI', 'BUS_UCLM_2025_V3::FLBA', 'BUS_UCLM_2025_V3::FUHI', 'BUS_UCLM_2025_V3::HUBL', 'BUS_UCLM_2025_V3::MENE']
No held-out performance has been computed or displayed.


In [3]:
# @title 11E-R-2. Extract or resume the fixed canonical ResNet-50 V2-weight embeddings
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision.models import resnet50, ResNet50_Weights
from tqdm.auto import tqdm

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCH_SIZE = BATCH_SIZE_GPU if DEVICE.type == 'cuda' else BATCH_SIZE_CPU
WEIGHTS = ResNet50_Weights.IMAGENET1K_V2
TRANSFORM = WEIGHTS.transforms(antialias=True)
MODEL = resnet50(weights=WEIGHTS)
MODEL.fc = nn.Identity()
MODEL.eval().to(DEVICE)
EMBEDDING_DIMENSION = 2048

def model_state_sha256(model):
    h = hashlib.sha256()
    for name, tensor in sorted(model.state_dict().items()):
        h.update(name.encode()+b'\0')
        array = tensor.detach().cpu().contiguous().numpy()
        h.update(str(array.dtype).encode()+b'\0')
        h.update(np.asarray(array.shape, dtype=np.int64).tobytes())
        h.update(array.tobytes(order='C'))
    return h.hexdigest()

MODEL_STATE_SHA256 = model_state_sha256(MODEL)

class FrozenVirtualImageDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image_bytes = read_virtual(row.dataset_id, row.image_virtual_path)
        if sha_bytes(image_bytes) != row.raw_image_sha256:
            raise RuntimeError(f'Raw source-image hash changed: {row.sample_id}')
        with Image.open(io.BytesIO(image_bytes)) as image0:
            image = ImageOps.exif_transpose(image0).convert('RGB')
            tensor = TRANSFORM(image)
        return tensor, int(index)

def embedding_paths(dataset_id):
    stem = f'{dataset_id}_Canonical_ResNet50_IMAGENET1K_V2_L2'
    return {
        'embeddings':P1/f'{stem}_Embeddings_v0.1.npy',
        'sample_ids':P1/f'{stem}_SampleIDs_v0.1.npy',
        'completion':P1/f'{stem}_Completion_v0.1.npy',
    }

def initialise_or_open_arrays(dataset_id, frame):
    paths = embedding_paths(dataset_id)
    n = len(frame)
    ids = frame.sample_id.astype(str).to_numpy()
    width = max(1, max(map(len, ids)))
    if not paths['embeddings'].exists():
        emb = np.lib.format.open_memmap(paths['embeddings'], mode='w+', dtype=np.float32, shape=(n, EMBEDDING_DIMENSION))
        emb[:] = 0
        emb.flush()
        saved_ids = np.lib.format.open_memmap(paths['sample_ids'], mode='w+', dtype=f'<U{width}', shape=(n,))
        saved_ids[:] = ids
        saved_ids.flush()
        done = np.lib.format.open_memmap(paths['completion'], mode='w+', dtype=np.bool_, shape=(n,))
        done[:] = False
        done.flush()
    emb = np.lib.format.open_memmap(paths['embeddings'], mode='r+')
    saved_ids = np.load(paths['sample_ids'], allow_pickle=False)
    done = np.lib.format.open_memmap(paths['completion'], mode='r+')
    assert emb.shape == (n, EMBEDDING_DIMENSION) and emb.dtype == np.float32
    assert done.shape == (n,) and np.array_equal(saved_ids.astype(str), ids)
    return paths, emb, done

embedding_rows = []
for dataset_id in SOURCE_DATASETS:
    frame = split_manifest[split_manifest.dataset_id==dataset_id].sort_values('sample_id').reset_index(drop=True)
    paths, embeddings_mm, completion_mm = initialise_or_open_arrays(dataset_id, frame)
    pending = np.flatnonzero(~np.asarray(completion_mm, dtype=bool))
    if len(pending):
        dataset = FrozenVirtualImageDataset(frame)
        loader = DataLoader(
            Subset(dataset, pending.tolist()), batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda')
        )
        for step, (images, local_indices) in enumerate(tqdm(loader, desc=f'Embedding {dataset_id}')):
            images = images.to(DEVICE, non_blocking=True)
            with torch.inference_mode():
                features = F.normalize(MODEL(images), p=2, dim=1)
            indices = np.asarray(local_indices, dtype=np.int64)
            embeddings_mm[indices] = features.detach().cpu().numpy().astype(np.float32)
            completion_mm[indices] = True
            if (step+1) % 10 == 0:
                embeddings_mm.flush(); completion_mm.flush()
        embeddings_mm.flush(); completion_mm.flush()
    assert np.asarray(completion_mm, dtype=bool).all(), f'Incomplete embeddings: {dataset_id}'
    norms = np.linalg.norm(np.asarray(embeddings_mm), axis=1)
    assert np.isfinite(embeddings_mm).all() and np.max(np.abs(norms-1.0)) < 2e-5
    embedding_rows.append({
        'dataset_id':dataset_id,
        'images':len(frame),
        'dimension':EMBEDDING_DIMENSION,
        'dtype':'float32',
        'model':'torchvision_resnet50_IMAGENET1K_V2',
        'model_state_sha256':MODEL_STATE_SHA256,
        'embedding_sha256':sha_file(paths['embeddings']),
        'sample_ids_sha256':sha_file(paths['sample_ids']),
        'completion_sha256':sha_file(paths['completion']),
        'maximum_l2_norm_error':float(np.max(np.abs(norms-1.0))),
    })
    del embeddings_mm, completion_mm
    gc.collect()

embedding_manifest = pd.DataFrame(embedding_rows)
write_csv_once(EMBEDDING_MANIFEST, embedding_manifest)
environment_payload = {
    'stage':'Stage11E-R',
    'python':sys.version,
    'platform':platform.platform(),
    'numpy':np.__version__,
    'pandas':pd.__version__,
    'scikit_learn':sklearn.__version__,
    'torch':torch.__version__,
    'torchvision':torchvision.__version__,
    'device':str(DEVICE),
    'device_name':torch.cuda.get_device_name(0) if DEVICE.type=='cuda' else platform.processor(),
    'batch_size':BATCH_SIZE,
    'num_workers':NUM_WORKERS,
    'weights':'ResNet50_Weights.IMAGENET1K_V2',
    'model_state_sha256':MODEL_STATE_SHA256,
    'embedding_manifest_sha256':sha_file(EMBEDDING_MANIFEST),
}
if ENVIRONMENT.exists():
    recorded_environment = json.loads(ENVIRONMENT.read_text(encoding='utf-8'))
    assert recorded_environment['model_state_sha256'] == MODEL_STATE_SHA256
    assert recorded_environment['embedding_manifest_sha256'] == sha_file(EMBEDDING_MANIFEST)
else:
    write_json_once(ENVIRONMENT, environment_payload)
display(embedding_manifest)
print('Execution device:', DEVICE)
print('Frozen representation complete; held-out labels remain unobserved.')


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 170MB/s]


Embedding BUS_BRA_2024:   0%|          | 0/30 [00:00<?, ?it/s]

Embedding BUSI_WHU_2025_V3:   0%|          | 0/15 [00:00<?, ?it/s]

Embedding BUS_UCLM_2025_V3:   0%|          | 0/5 [00:00<?, ?it/s]

Embedding RODRIGUES_BUI_2017:   0%|          | 0/4 [00:00<?, ?it/s]

,dataset_id,images,dimension,dtype,model,model_state_sha256,embedding_sha256,sample_ids_sha256,completion_sha256,maximum_l2_norm_error
0,BUS_BRA_2024,1875,2048,float32,torchvision_resnet50_IMAGENET1K_V2,3f2c393680172fd552aae83bd2f0e3c389457e7d13499e...,8cf5db9f84eb465c91caf2be72899c1cbc3753060723d6...,947d9b897056102b788d7149c0d1e4dc98973031be8c1d...,e4f6525a1526670f8480ab0cf6876d3e165ffb871aea7a...,1.788139e-07
1,BUSI_WHU_2025_V3,926,2048,float32,torchvision_resnet50_IMAGENET1K_V2,3f2c393680172fd552aae83bd2f0e3c389457e7d13499e...,cbe7df46230033173565d38e9b0a1344a89e400ee10229...,1c85858eb1621fdc076cd4c5ad6e6cbfefe9759dd20db8...,e39270bf363b91b0e2681d311ead61a0860dd81ed84bf2...,1.192093e-07
2,BUS_UCLM_2025_V3,263,2048,float32,torchvision_resnet50_IMAGENET1K_V2,3f2c393680172fd552aae83bd2f0e3c389457e7d13499e...,3031b77ed6b035ff7bfc566f3303584e6a7a21636bbf65...,936c4af8ec0a8f61ed861cc58889251273bb19780d28ab...,001722d645d0034dd1bf8f758574d2e72019d0638d388e...,1.192093e-07
3,RODRIGUES_BUI_2017,242,2048,float32,torchvision_resnet50_IMAGENET1K_V2,3f2c393680172fd552aae83bd2f0e3c389457e7d13499e...,451828b6b04d65bbbfaf1b9f79816ff5cd3db1c3ffdec8...,ba3c46776824499ec2f4f9f79ac22f947a7e957eadc8ea...,639bf8c07304aaa8843592371302217beb3db44c83b086...,1.192093e-07


Execution device: cuda
Frozen representation complete; held-out labels remain unobserved.


In [4]:
# @title 11E-R-3. Compute development OOF evidence and freeze final development axes before validation
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def sigmoid(scores):
    return 1.0/(1.0+np.exp(-np.clip(np.asarray(scores, dtype=np.float64), -60, 60)))

def load_aligned_embeddings(dataset_id, frame):
    paths = embedding_paths(dataset_id)
    ids = np.load(paths['sample_ids'], allow_pickle=False).astype(str)
    embeddings = np.load(paths['embeddings'], mmap_mode='r', allow_pickle=False)
    lookup = {sample_id:index for index, sample_id in enumerate(ids)}
    assert len(lookup) == len(ids)
    indices = np.asarray([lookup[sample_id] for sample_id in frame.sample_id.astype(str)], dtype=np.int64)
    return np.asarray(embeddings[indices], dtype=np.float64)

def group_total_one_weights(groups):
    groups = pd.Series(np.asarray(groups, dtype=str))
    counts = groups.value_counts()
    weights = groups.map(lambda value: 1.0/counts[value]).to_numpy(dtype=np.float64)
    assert np.allclose(pd.DataFrame({'group':groups, 'weight':weights}).groupby('group').weight.sum().to_numpy(), 1.0)
    return weights

def new_probe(seed):
    return Pipeline([
        ('scaler', StandardScaler(copy=True, with_mean=True, with_std=True)),
        ('logisticregression', LogisticRegression(
            penalty='l2', C=LOGISTIC_C, solver='liblinear', class_weight='balanced',
            fit_intercept=True, max_iter=LOGISTIC_MAX_ITER, random_state=int(seed)
        )),
    ])

def fit_probe(x, y, groups, seed):
    probe = new_probe(seed)
    weights = group_total_one_weights(groups)
    probe.fit(x, y, logisticregression__sample_weight=weights)
    assert probe.named_steps['logisticregression'].n_iter_[0] < LOGISTIC_MAX_ITER
    return probe

def group_bootstrap_auc(y, scores, groups, seed):
    y = np.asarray(y, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    groups = np.asarray(groups, dtype=str)
    unique_groups = np.unique(groups)
    group_indices = {group:np.flatnonzero(groups==group) for group in unique_groups}
    rng = np.random.default_rng(seed)
    aucs = []
    attempts = 0
    while len(aucs) < N_BOOTSTRAP and attempts < N_BOOTSTRAP*20:
        sampled_groups = rng.choice(unique_groups, size=len(unique_groups), replace=True)
        indices = np.concatenate([group_indices[group] for group in sampled_groups])
        attempts += 1
        if np.unique(y[indices]).size < 2:
            continue
        aucs.append(roc_auc_score(y[indices], scores[indices]))
    assert len(aucs) == N_BOOTSTRAP, f'Only {len(aucs)} valid bootstrap replicates'
    lower, upper = np.quantile(np.asarray(aucs), [0.025,0.975])
    return float(lower), float(upper), attempts-N_BOOTSTRAP

def metric_row(dataset_id, partition, y, scores, groups, seed):
    y = np.asarray(y, dtype=np.int64)
    scores = np.asarray(scores, dtype=np.float64)
    groups = np.asarray(groups, dtype=str)
    assert len(y) == len(scores) == len(groups)
    assert np.unique(y).size == 2 and np.isfinite(scores).all()
    endpoint_composition = pd.DataFrame({'binary_label':y, 'group_id':groups}).groupby('group_id').binary_label.nunique()
    auc = float(roc_auc_score(y, scores))
    lower, upper, rejected = group_bootstrap_auc(y, scores, groups, seed)
    return {
        'dataset_id':dataset_id,
        'partition':partition,
        'images':len(y),
        'groups':len(np.unique(groups)),
        'mixed_endpoint_groups':int((endpoint_composition>1).sum()),
        'negative_images':int((y==0).sum()),
        'positive_images':int((y==1).sum()),
        'roc_auc':auc,
        'ci95_lower':lower,
        'ci95_upper':upper,
        'bootstrap_replicates':N_BOOTSTRAP,
        'rejected_single_class_draws':rejected,
        'metric_unit':'image_level_roc_auc',
        'bootstrap_unit':'released_group_cluster',
    }

def write_npz_once(path, arrays):
    path = Path(path)
    normalised = {key:np.asarray(value) for key,value in arrays.items()}
    if path.exists():
        with np.load(path, allow_pickle=False) as old:
            assert set(old.files) == set(normalised), f'Axis schema changed: {path}'
            for key, expected in normalised.items():
                observed = old[key]
                if observed.dtype.kind in 'fc':
                    assert np.allclose(observed, expected, rtol=0, atol=1e-12), f'Axis changed: {path} :: {key}'
                else:
                    assert np.array_equal(observed, expected), f'Axis changed: {path} :: {key}'
        return
    temporary = path.with_name(path.name+'.tmp.npz')
    np.savez_compressed(temporary, **normalised)
    temporary.replace(path)

oof_prediction_rows, oof_metric_rows, axis_rows = [], [], []
axis_paths = {}

for source_index, dataset_id in enumerate(SOURCE_DATASETS):
    development = split_manifest[(split_manifest.dataset_id==dataset_id)&(split_manifest.partition=='development')].copy()
    development = development.sort_values('sample_id').reset_index(drop=True)
    x = load_aligned_embeddings(dataset_id, development)
    y = development.binary_label.astype(int).to_numpy()
    groups = development.group_id.astype(str).to_numpy()
    folds = development.oof_fold.astype(int).to_numpy()
    unique_folds = sorted(np.unique(folds).tolist())
    assert len(unique_folds) >= 3 and unique_folds == list(range(len(unique_folds)))
    oof_scores = np.full(len(development), np.nan, dtype=np.float64)

    for fold in unique_folds:
        train_mask, validation_mask = folds!=fold, folds==fold
        assert np.unique(y[train_mask]).size == 2 and np.unique(y[validation_mask]).size == 2
        probe = fit_probe(x[train_mask], y[train_mask], groups[train_mask], RANDOM_SEED+source_index*100+fold)
        oof_scores[validation_mask] = probe.decision_function(x[validation_mask])
    assert np.isfinite(oof_scores).all()
    metric = metric_row(
        dataset_id, 'development_oof', y, oof_scores, groups,
        RANDOM_SEED+source_index*100+1
    )
    metric['oof_folds'] = len(unique_folds)
    oof_metric_rows.append(metric)
    for row, score in zip(development.itertuples(), oof_scores):
        oof_prediction_rows.append({
            'dataset_id':dataset_id, 'sample_id':row.sample_id, 'group_id':row.group_id,
            'binary_label':int(row.binary_label), 'oof_fold':int(row.oof_fold),
            'decision_score':float(score), 'probability':float(sigmoid(score)),
        })

    final_probe = fit_probe(x, y, groups, RANDOM_SEED+source_index*100+99)
    scaler = final_probe.named_steps['scaler']
    classifier = final_probe.named_steps['logisticregression']
    coefficient_standardised = classifier.coef_.reshape(-1).astype(np.float64)
    intercept_standardised = float(classifier.intercept_[0])
    coefficient_raw = coefficient_standardised/scaler.scale_.astype(np.float64)
    intercept_raw = intercept_standardised-float(np.dot(scaler.mean_.astype(np.float64), coefficient_raw))
    pipeline_scores = final_probe.decision_function(x)
    raw_scores = x@coefficient_raw+intercept_raw
    maximum_error = float(np.max(np.abs(pipeline_scores-raw_scores)))
    assert maximum_error < AXIS_EQUIVALENCE_TOLERANCE
    axis_path = P3/f'{dataset_id}_Frozen_Development_Source_Axis_v0.1.npz'
    axis_paths[dataset_id] = axis_path
    arrays = {
        'dataset_id':np.asarray(dataset_id),
        'representation':np.asarray('torchvision_resnet50_IMAGENET1K_V2_L2'),
        'model_state_sha256':np.asarray(MODEL_STATE_SHA256),
        'scaler_mean':scaler.mean_.astype(np.float64),
        'scaler_scale':scaler.scale_.astype(np.float64),
        'coefficient_standardised':coefficient_standardised,
        'intercept_standardised':np.asarray(intercept_standardised, dtype=np.float64),
        'coefficient_raw':coefficient_raw,
        'intercept_raw':np.asarray(intercept_raw, dtype=np.float64),
        'logistic_C':np.asarray(LOGISTIC_C, dtype=np.float64),
        'random_seed':np.asarray(RANDOM_SEED+source_index*100+99, dtype=np.int64),
        'development_images':np.asarray(len(development), dtype=np.int64),
        'development_groups':np.asarray(len(np.unique(groups)), dtype=np.int64),
        'axis_frozen_before_heldout_validation':np.asarray(True),
    }
    write_npz_once(axis_path, arrays)
    axis_rows.append({
        'dataset_id':dataset_id,
        'development_images':len(development),
        'development_groups':len(np.unique(groups)),
        'dimension':len(coefficient_raw),
        'raw_axis_l2_norm':float(np.linalg.norm(coefficient_raw)),
        'intercept_raw':intercept_raw,
        'maximum_pipeline_raw_axis_error':maximum_error,
        'axis_frozen_before_heldout_validation':True,
        'axis_npz_sha256':sha_file(axis_path),
    })

oof_predictions = pd.DataFrame(oof_prediction_rows).sort_values(['dataset_id','sample_id']).reset_index(drop=True)
oof_summary = pd.DataFrame(oof_metric_rows).sort_values('dataset_id').reset_index(drop=True)
axis_manifest = pd.DataFrame(axis_rows).sort_values('dataset_id').reset_index(drop=True)
write_csv_once(OOF_PREDICTIONS, oof_predictions)
write_csv_once(OOF_SUMMARY, oof_summary)
write_csv_once(AXIS_MANIFEST, axis_manifest)

freeze_payload = {
    'stage':'Stage11E-R',
    'event':'PREVALIDATION_SOURCE_AXIS_FREEZE',
    'protocol_seal_sha256':protocol['seal_sha256'],
    'parent_stage11d_r_handoff_sha256':parent_handoff['handoff_sha256'],
    'embedding_manifest_sha256':sha_file(EMBEDDING_MANIFEST),
    'development_oof_predictions_sha256':sha_file(OOF_PREDICTIONS),
    'development_oof_summary_sha256':sha_file(OOF_SUMMARY),
    'axis_manifest_sha256':sha_file(AXIS_MANIFEST),
    'axis_sha256_by_dataset':{row.dataset_id:row.axis_npz_sha256 for row in axis_manifest.itertuples()},
    'source_dataset_ids':SOURCE_DATASETS,
    'heldout_performance_observed_before_axis_freeze':False,
    'threshold_or_model_tuning_performed':False,
    'stage12_authorised':False,
    'ddo2_fitted':False,
    'locked_blind_assets_touched':False,
}
axis_freeze = create_or_verify_seal(AXIS_FREEZE, freeze_payload, 'freeze_sha256', 'frozen_utc')
display(oof_summary)
display(axis_manifest)
print('Pre-validation axis freeze:', axis_freeze['freeze_sha256'])
print('Held-out source performance observed before freeze:', axis_freeze['heldout_performance_observed_before_axis_freeze'])


,dataset_id,partition,images,groups,mixed_endpoint_groups,negative_images,positive_images,roc_auc,ci95_lower,ci95_upper,bootstrap_replicates,rejected_single_class_draws,metric_unit,bootstrap_unit,oof_folds
0,BUSI_WHU_2025_V3,development_oof,740,652,0,447,293,0.810729,0.770312,0.843025,2000,0,image_level_roc_auc,released_group_cluster,5
1,BUS_BRA_2024,development_oof,1498,852,0,1013,485,0.739954,0.708219,0.769169,2000,0,image_level_roc_auc,released_group_cluster,5
2,BUS_UCLM_2025_V3,development_oof,198,29,4,131,67,0.688390,0.563970,0.786914,2000,0,image_level_roc_auc,released_group_cluster,5
3,RODRIGUES_BUI_2017,development_oof,194,194,0,77,117,0.971806,0.947366,0.991624,2000,0,image_level_roc_auc,released_group_cluster,5


,dataset_id,development_images,development_groups,dimension,raw_axis_l2_norm,intercept_raw,maximum_pipeline_raw_axis_error,axis_frozen_before_heldout_validation,axis_npz_sha256
0,BUSI_WHU_2025_V3,740,652,2048,2904.861427,-1.806255,2.131628e-14,True,7d09dd72c43d9dc43d574e8d6dea90edd7e5fbb845103d...
1,BUS_BRA_2024,1498,852,2048,4086.415956,-3.729191,6.039613e-14,True,73ad923e912177b56ff21cab081b0b84e4ae30a37f0f7b...
2,BUS_UCLM_2025_V3,198,29,2048,4404.535824,-0.631567,1.421085e-14,True,3fb6eedf6d08dac78d24549e40c8ffcae835c1c37fcf0b...
3,RODRIGUES_BUI_2017,194,194,2048,4962.638640,4.142952,1.421085e-14,True,6f09b1002f3b1928aa2f957e86887213c58e330b5c6eef...


Pre-validation axis freeze: 98903477c84b7ec6b33258e30f02e8f357ec36e4261c6cbd242fb73ba8c1ec20
Held-out source performance observed before freeze: False


In [5]:
# @title 11E-R-4. Observe held-out source validation once and apply the frozen recoverability gate
axis_freeze = verify_self(AXIS_FREEZE, 'freeze_sha256')
assert axis_freeze['heldout_performance_observed_before_axis_freeze'] is False
assert axis_freeze['axis_manifest_sha256'] == sha_file(AXIS_MANIFEST)

heldout_prediction_rows, heldout_metric_rows = [], []
for source_index, dataset_id in enumerate(SOURCE_DATASETS):
    heldout = split_manifest[(split_manifest.dataset_id==dataset_id)&(split_manifest.partition=='heldout')].copy()
    heldout = heldout.sort_values('sample_id').reset_index(drop=True)
    x = load_aligned_embeddings(dataset_id, heldout)
    y = heldout.binary_label.astype(int).to_numpy()
    groups = heldout.group_id.astype(str).to_numpy()
    with np.load(axis_paths[dataset_id], allow_pickle=False) as axis:
        assert str(axis['dataset_id']) == dataset_id
        assert str(axis['model_state_sha256']) == MODEL_STATE_SHA256
        assert bool(axis['axis_frozen_before_heldout_validation']) is True
        scores = x@axis['coefficient_raw'].astype(np.float64)+float(axis['intercept_raw'])
    metric = metric_row(
        dataset_id, 'heldout', y, scores, groups,
        RANDOM_SEED+source_index*100+2
    )
    heldout_metric_rows.append(metric)
    for row, score in zip(heldout.itertuples(), scores):
        heldout_prediction_rows.append({
            'dataset_id':dataset_id, 'sample_id':row.sample_id, 'group_id':row.group_id,
            'binary_label':int(row.binary_label), 'decision_score':float(score),
            'probability':float(sigmoid(score)),
        })

runtime['heldout_validation_observed'] = True
heldout_predictions = pd.DataFrame(heldout_prediction_rows).sort_values(['dataset_id','sample_id']).reset_index(drop=True)
heldout_summary = pd.DataFrame(heldout_metric_rows).sort_values('dataset_id').reset_index(drop=True)
write_csv_once(HELDOUT_PREDICTIONS, heldout_predictions)
write_csv_once(HELDOUT_SUMMARY, heldout_summary)

recoverability_rows = []
for dataset_id in SOURCE_DATASETS:
    development = oof_summary[oof_summary.dataset_id==dataset_id].iloc[0]
    heldout = heldout_summary[heldout_summary.dataset_id==dataset_id].iloc[0]
    passed = bool(
        development.roc_auc >= AUC_THRESHOLD and development.ci95_lower > CI_LOWER_THRESHOLD and
        heldout.roc_auc >= AUC_THRESHOLD and heldout.ci95_lower > CI_LOWER_THRESHOLD
    )
    recoverability_rows.append({
        'dataset_id':dataset_id,
        'development_oof_auc':float(development.roc_auc),
        'development_oof_ci95_lower':float(development.ci95_lower),
        'development_oof_ci95_upper':float(development.ci95_upper),
        'heldout_auc':float(heldout.roc_auc),
        'heldout_ci95_lower':float(heldout.ci95_lower),
        'heldout_ci95_upper':float(heldout.ci95_upper),
        'auc_threshold':AUC_THRESHOLD,
        'ci_lower_strict_threshold':CI_LOWER_THRESHOLD,
        'source_recoverability_pass':passed,
        'source_status':'PASS_RECOVERABLE_SOURCE' if passed else 'RETIRED_SOURCE_RECOVERABILITY_GATE_FAILED',
        'outgoing_development_edges_authorised':passed,
        'failed_source_rescue_performed':False,
    })

recoverability_summary = pd.DataFrame(recoverability_rows).sort_values('dataset_id').reset_index(drop=True)
PASSING_SOURCES = recoverability_summary.loc[recoverability_summary.source_recoverability_pass,'dataset_id'].tolist()
RETIRED_SOURCES = recoverability_summary.loc[~recoverability_summary.source_recoverability_pass,'dataset_id'].tolist()
GLOBAL_SOURCE_GATE_PASSED = len(PASSING_SOURCES) >= MINIMUM_RECOVERABLE_SOURCES
edge_rows = [
    {'source_dataset_id':source, 'target_dataset_id':target, 'edge_id':f'{source}__TO__{target}', 'development_only':True, 'directed_transfer_outcome_not_evaluated_in_stage11e_r':True}
    for source in PASSING_SOURCES for target in SOURCE_DATASETS if target != source
]
edge_roster = pd.DataFrame(edge_rows, columns=['source_dataset_id','target_dataset_id','edge_id','development_only','directed_transfer_outcome_not_evaluated_in_stage11e_r'])
write_csv_once(RECOVERABILITY_SUMMARY, recoverability_summary)
write_csv_once(EDGE_ROSTER, edge_roster)

decision = (
    'SEAL_STAGE11E_R_AUTHORISE_STAGE11F_R_DEVELOPMENT_ONLY_DIRECTED_EDGE_ASSEMBLY_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED'
    if GLOBAL_SOURCE_GATE_PASSED else
    'SEAL_STAGE11E_R_HOLD_GLOBAL_EDGE_GATE_INSUFFICIENT_RECOVERABLE_SOURCES_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED'
)
handoff_payload = {
    'stage':'Stage11E-R',
    'target':'Stage11F-R',
    'decision':decision,
    'protocol_seal_sha256':protocol['seal_sha256'],
    'parent_stage11d_r_final_sha256':parent_final['final_record_sha256'],
    'prevalidation_axis_freeze_sha256':axis_freeze['freeze_sha256'],
    'embedding_manifest_sha256':sha_file(EMBEDDING_MANIFEST),
    'development_oof_summary_sha256':sha_file(OOF_SUMMARY),
    'axis_manifest_sha256':sha_file(AXIS_MANIFEST),
    'heldout_summary_sha256':sha_file(HELDOUT_SUMMARY),
    'source_recoverability_summary_sha256':sha_file(RECOVERABILITY_SUMMARY),
    'authorised_edge_roster_sha256':sha_file(EDGE_ROSTER),
    'evaluated_source_dataset_ids':SOURCE_DATASETS,
    'recoverable_source_dataset_ids':PASSING_SOURCES,
    'retired_source_dataset_ids':RETIRED_SOURCES,
    'recoverable_source_count':len(PASSING_SOURCES),
    'minimum_recoverable_sources':MINIMUM_RECOVERABLE_SOURCES,
    'global_source_gate_passed':GLOBAL_SOURCE_GATE_PASSED,
    'authorised_development_edge_count':len(edge_roster),
    'stage11f_r_development_only_authorised':GLOBAL_SOURCE_GATE_PASSED,
    'source_gate':'development OOF and held-out image-level AUC each >=0.70 with released-group cluster-bootstrap lower 95% CI >0.55',
    'failed_source_rescue_performed':False,
    'transfer_outcomes_evaluated':False,
    'stage12_authorised':False,
    'ddo2_fitted':False,
    'locked_blind_assets_touched':False,
}
handoff = create_or_verify_seal(HANDOFF, handoff_payload, 'handoff_sha256', 'sealed_utc')
display(recoverability_summary)
print('Passing sources:', PASSING_SOURCES)
print('Retired sources:', RETIRED_SOURCES)
print('Global source gate:', GLOBAL_SOURCE_GATE_PASSED, f'({len(PASSING_SOURCES)}/{MINIMUM_RECOVERABLE_SOURCES})')
print('Decision:', decision)


,dataset_id,development_oof_auc,development_oof_ci95_lower,development_oof_ci95_upper,heldout_auc,heldout_ci95_lower,heldout_ci95_upper,auc_threshold,ci_lower_strict_threshold,source_recoverability_pass,source_status,outgoing_development_edges_authorised,failed_source_rescue_performed
0,BUSI_WHU_2025_V3,0.810729,0.770312,0.843025,0.801520,0.727751,0.864459,0.7,0.55,True,PASS_RECOVERABLE_SOURCE,True,False
1,BUS_BRA_2024,0.739954,0.708219,0.769169,0.721986,0.660696,0.783547,0.7,0.55,True,PASS_RECOVERABLE_SOURCE,True,False
2,BUS_UCLM_2025_V3,0.688390,0.563970,0.786914,0.665962,0.321894,0.795272,0.7,0.55,False,RETIRED_SOURCE_RECOVERABILITY_GATE_FAILED,False,False
3,RODRIGUES_BUI_2017,0.971806,0.947366,0.991624,0.990926,0.969643,1.000000,0.7,0.55,True,PASS_RECOVERABLE_SOURCE,True,False


Passing sources: ['BUSI_WHU_2025_V3', 'BUS_BRA_2024', 'RODRIGUES_BUI_2017']
Retired sources: ['BUS_UCLM_2025_V3']
Global source gate: True (3/3)
Decision: SEAL_STAGE11E_R_AUTHORISE_STAGE11F_R_DEVELOPMENT_ONLY_DIRECTED_EDGE_ASSEMBLY_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED


In [6]:
# @title 11E-R-5. Run independent leakage, chronology, metric, axis, and firewall checks
checks = []
def check(name, passed, evidence):
    checks.append({'check':name, 'passed':bool(passed), 'evidence':str(evidence)[:2000]})

check('Stage11D-R protocol exact', parent_protocol['seal_sha256']==EXPECTED_PARENT_PROTOCOL, parent_protocol['seal_sha256'])
check('Stage11D-R exact manifest exact', sha_file(PARENT_EXACT)==EXPECTED_PARENT_EXACT, sha_file(PARENT_EXACT))
check('Stage11D-R dedup summary exact', sha_file(PARENT_DEDUP)==EXPECTED_PARENT_DEDUP, sha_file(PARENT_DEDUP))
check('Stage11D-R grouped split cross-hashes exact', sha_file(PARENT_SPLITS)==parent_handoff['split_manifest_sha256']==parent_final['split_manifest_sha256'], sha_file(PARENT_SPLITS))
check('four frozen source domains exact', len(SOURCE_DATASETS)==EXPECTED_SPLIT_READY_DOMAINS, SOURCE_DATASETS)
check('retained image count exact', len(split_manifest)==EXPECTED_RETAINED_IMAGES, len(split_manifest))
check('all released groups stay in one partition', split_manifest.groupby(['dataset_id','group_id']).partition.nunique().max()==1, len(group_endpoint_audit))
check('known mixed endpoint groups audited exactly', set(mixed_endpoint_group_ids)==set(EXPECTED_MIXED_ENDPOINT_GROUP_IDS), sorted(mixed_endpoint_group_ids))
check('fixed representation one model hash', embedding_manifest.model_state_sha256.nunique()==1 and embedding_manifest.model_state_sha256.iloc[0]==MODEL_STATE_SHA256, MODEL_STATE_SHA256)
check('all embeddings complete', all(np.load(embedding_paths(d)['completion'],allow_pickle=False).all() for d in SOURCE_DATASETS), SOURCE_DATASETS)
check('all embeddings finite and L2 normalised', embedding_manifest.maximum_l2_norm_error.max()<2e-5, embedding_manifest.maximum_l2_norm_error.max())
check('development groups stay in one OOF fold', development_groups.groupby(['dataset_id','group_id']).oof_fold.nunique().max()==1, len(development_groups))
check('every development image has one OOF prediction', len(oof_predictions)==int((split_manifest.partition=='development').sum()) and oof_predictions.sample_id.is_unique, len(oof_predictions))
check('no heldout sample in OOF predictions', not set(oof_predictions.sample_id)&set(split_manifest.loc[split_manifest.partition=='heldout','sample_id']), 'empty intersection')
check('all OOF predictions finite', np.isfinite(oof_predictions.decision_score).all(), len(oof_predictions))
oof_auc_readback = {dataset_id:float(roc_auc_score(frame.binary_label.astype(int), frame.decision_score.astype(float))) for dataset_id,frame in oof_predictions.groupby('dataset_id')}
check('OOF point estimates are exact image-level AUC', all(np.isclose(oof_auc_readback[row.dataset_id], row.roc_auc, rtol=0, atol=1e-12) for row in oof_summary.itertuples()), oof_auc_readback)
check('OOF uncertainty unit is released-group cluster', (oof_summary.metric_unit=='image_level_roc_auc').all() and (oof_summary.bootstrap_unit=='released_group_cluster').all(), oof_summary[['metric_unit','bootstrap_unit']].drop_duplicates().to_dict('records'))
check('all source axes frozen before heldout validation', axis_manifest.axis_frozen_before_heldout_validation.all(), axis_manifest.dataset_id.tolist())
check('axis freeze predates validation performance observation by protocol state', axis_freeze['heldout_performance_observed_before_axis_freeze'] is False and runtime['heldout_validation_observed'] is True, axis_freeze['freeze_sha256'])
check('pipeline and raw axes equivalent', axis_manifest.maximum_pipeline_raw_axis_error.max()<AXIS_EQUIVALENCE_TOLERANCE, axis_manifest.maximum_pipeline_raw_axis_error.max())
check('every heldout image scored once', len(heldout_predictions)==int((split_manifest.partition=='heldout').sum()) and heldout_predictions.sample_id.is_unique, len(heldout_predictions))
check('all heldout predictions finite', np.isfinite(heldout_predictions.decision_score).all(), len(heldout_predictions))
heldout_auc_readback = {dataset_id:float(roc_auc_score(frame.binary_label.astype(int), frame.decision_score.astype(float))) for dataset_id,frame in heldout_predictions.groupby('dataset_id')}
check('heldout point estimates are exact image-level AUC', all(np.isclose(heldout_auc_readback[row.dataset_id], row.roc_auc, rtol=0, atol=1e-12) for row in heldout_summary.itertuples()), heldout_auc_readback)
check('heldout uncertainty unit is released-group cluster', (heldout_summary.metric_unit=='image_level_roc_auc').all() and (heldout_summary.bootstrap_unit=='released_group_cluster').all(), heldout_summary[['metric_unit','bootstrap_unit']].drop_duplicates().to_dict('records'))
check('source gate applied without rescue', (~recoverability_summary.failed_source_rescue_performed).all(), recoverability_summary.source_status.tolist())
check('outgoing edge sources all passed', set(edge_roster.source_dataset_id).issubset(PASSING_SOURCES) if len(edge_roster) else True, PASSING_SOURCES)
check('global edge gate exact', GLOBAL_SOURCE_GATE_PASSED==(len(PASSING_SOURCES)>=MINIMUM_RECOVERABLE_SOURCES), len(PASSING_SOURCES))
check('no mask used as source image', not split_manifest.image_virtual_path.astype(str).str.contains(r'(?:^|[/!])(?:masks?|gt)(?:[/!]|$)|_tumou?r\.',case=False,regex=True).any(), 'source paths audited')
check('no locked-blind token in image paths', not any(token.lower() in inventory_text for token in LOCKED), LOCKED)
check('no transfer outcome evaluated', runtime['transfer_outcomes_evaluated'] is False and handoff['transfer_outcomes_evaluated'] is False, False)
check('DDO2 and Stage12 prohibited', runtime['ddo2_fitted'] is False and handoff['ddo2_fitted'] is False and handoff['stage12_authorised'] is False, False)
check('locked blind untouched', runtime['locked_blind_assets_touched'] is False and handoff['locked_blind_assets_touched'] is False, False)
check('handoff self hash', sha_json({k:v for k,v in handoff.items() if k!='handoff_sha256'})==handoff['handoff_sha256'], handoff['handoff_sha256'])

validity = pd.DataFrame(checks)
write_csv_once(FIREWALL, validity)
failed = validity.loc[~validity.passed, 'check'].tolist()
assert not failed, 'Validity failure: '+'; '.join(failed)
print('Independent checks:', int(validity.passed.sum()), '/', len(validity), 'passed')


Independent checks: 33 / 33 passed


In [7]:
# @title 11E-R-6. Seal the report, output manifest, final record, and next decision gate
report = f"""# Stage 11E-R report

## Answer first

- Evaluated split-ready development sources: **{len(SOURCE_DATASETS)}** — {', '.join(SOURCE_DATASETS)}.
- Frozen representation: **torchvision ResNet-50 / IMAGENET1K_V2 / 2,048-D L2-normalised embedding**.
- Frozen source gate: **development OOF and held-out image-level AUC each >= {AUC_THRESHOLD:.2f}; both released-group cluster-bootstrap 95% CI lower bounds > {CI_LOWER_THRESHOLD:.2f}**.
- Mixed-endpoint released groups: **{len(mixed_endpoint_group_ids)}**, retained intact and audited rather than relabelled or split.
- Recoverable sources: **{len(PASSING_SOURCES)}/{MINIMUM_RECOVERABLE_SOURCES} required** — {', '.join(PASSING_SOURCES) if PASSING_SOURCES else 'none'}.
- Retired sources: **{', '.join(RETIRED_SOURCES) if RETIRED_SOURCES else 'none'}**.
- Authorised development-only directed edges: **{len(edge_roster)}**.
- Global source gate passed: **{GLOBAL_SOURCE_GATE_PASSED}**.
- Decision: `{decision}`.

## Source recoverability

{markdown_table(recoverability_summary)}

## Frozen source axes

{markdown_table(axis_manifest)}

## Interpretation boundary

Passing this stage means only that a released image/lesion endpoint is recoverable from the fixed representation under the pre-specified grouped OOF and held-out gate. Point estimates are image-level ROC AUCs; uncertainty resamples intact released groups as clusters. It does not show that any source transfers to another dataset, that DDO predicts transfer risk, or that a general cross-modal observability law exists. Failed sources are retired without tuning or rescue in this protocol.

Stage 11E-R accessed only the four Stage 11D-R development domains. It did not evaluate a directed transfer edge, fit DDO-2, operate Stage 12, or access any locked-blind asset.
"""
write_text_once(REPORT, report)

dynamic_files = []
for dataset_id in SOURCE_DATASETS:
    paths = embedding_paths(dataset_id)
    dynamic_files.extend([paths['embeddings'], paths['sample_ids'], paths['completion'], axis_paths[dataset_id]])
tracked = [
    PROTOCOL,ABORTED_PROTOCOL,PARENT_COMMIT,ENVIRONMENT,SOURCE_INVENTORY,GROUP_ENDPOINT_AUDIT,EMBEDDING_MANIFEST,
    OOF_PREDICTIONS,OOF_SUMMARY,AXIS_MANIFEST,AXIS_FREEZE,
    HELDOUT_PREDICTIONS,HELDOUT_SUMMARY,RECOVERABILITY_SUMMARY,EDGE_ROSTER,HANDOFF,
    FIREWALL,REPORT,
] + dynamic_files
output_manifest = pd.DataFrame([
    {'relative_path':str(path.relative_to(ROOT)), 'size_bytes':path.stat().st_size, 'sha256':sha_file(path)}
    for path in tracked
]).sort_values('relative_path').reset_index(drop=True)
write_csv_once(OUTPUT_MANIFEST, output_manifest)

next_step = (
    'BUILD_STAGE11F_R_DEVELOPMENT_ONLY_DIRECTED_EDGE_ASSEMBLY_FROM_RECOVERABLE_FROZEN_SOURCE_AXES'
    if GLOBAL_SOURCE_GATE_PASSED else
    'HOLD_GLOBAL_EDGE_GATE_AND_REPORT_FIXED_PROTOCOL_SOURCE_FAILURES_WITHOUT_RESCUE'
)
final_payload = {
    'stage':'Stage11E-R',
    'version':'0.1',
    'decision':decision,
    'protocol_seal_sha256':protocol['seal_sha256'],
    'parent_stage11d_r_final_sha256':parent_final['final_record_sha256'],
    'parent_stage11d_r_handoff_sha256':parent_handoff['handoff_sha256'],
    'embedding_manifest_sha256':sha_file(EMBEDDING_MANIFEST),
    'development_oof_summary_sha256':sha_file(OOF_SUMMARY),
    'prevalidation_axis_freeze_sha256':axis_freeze['freeze_sha256'],
    'axis_manifest_sha256':sha_file(AXIS_MANIFEST),
    'heldout_summary_sha256':sha_file(HELDOUT_SUMMARY),
    'source_recoverability_summary_sha256':sha_file(RECOVERABILITY_SUMMARY),
    'authorised_edge_roster_sha256':sha_file(EDGE_ROSTER),
    'stage11f_r_handoff_sha256':handoff['handoff_sha256'],
    'firewall_sha256':sha_file(FIREWALL),
    'output_integrity_manifest_sha256':sha_file(OUTPUT_MANIFEST),
    'evaluated_source_count':len(SOURCE_DATASETS),
    'recoverable_source_count':len(PASSING_SOURCES),
    'recoverable_source_dataset_ids':PASSING_SOURCES,
    'retired_source_dataset_ids':RETIRED_SOURCES,
    'minimum_recoverable_sources':MINIMUM_RECOVERABLE_SOURCES,
    'global_source_gate_passed':GLOBAL_SOURCE_GATE_PASSED,
    'authorised_development_edge_count':len(edge_roster),
    'stage11f_r_development_only_authorised':GLOBAL_SOURCE_GATE_PASSED,
    'failed_source_rescue_performed':False,
    'transfer_outcomes_evaluated':False,
    'stage12_authorised':False,
    'ddo2_fitted':False,
    'locked_blind_assets_touched':False,
    'next_step':next_step,
}
final = create_or_verify_seal(FINAL, final_payload, 'final_record_sha256', 'completed_utc')
runtime.update({
    'completed':True,
    'decision':decision,
    'final_record_sha256':final['final_record_sha256'],
    'last_updated_utc':now(),
})
RUNTIME.write_text(json.dumps(runtime, indent=2, ensure_ascii=False)+'\n', encoding='utf-8')

print('================ STAGE 11E-R COMPLETE ================')
print('Evaluated sources / recoverable / minimum:', final['evaluated_source_count'], '/', final['recoverable_source_count'], '/', final['minimum_recoverable_sources'])
print('Recoverable sources:', final['recoverable_source_dataset_ids'])
print('Retired sources:', final['retired_source_dataset_ids'])
print('Authorised development-only directed edges:', final['authorised_development_edge_count'])
print('Global source gate passed:', final['global_source_gate_passed'])
print('Decision:', final['decision'])
print('Protocol seal:', final['protocol_seal_sha256'])
print('Embedding manifest hash:', final['embedding_manifest_sha256'])
print('Axis freeze hash:', final['prevalidation_axis_freeze_sha256'])
print('Recoverability summary hash:', final['source_recoverability_summary_sha256'])
print('Stage11F-R handoff hash:', final['stage11f_r_handoff_sha256'])
print('Final record hash:', final['final_record_sha256'])
print('Next step:', final['next_step'])


================ STAGE 11E-R COMPLETE ================
Evaluated sources / recoverable / minimum: 4 / 3 / 3
Recoverable sources: ['BUSI_WHU_2025_V3', 'BUS_BRA_2024', 'RODRIGUES_BUI_2017']
Retired sources: ['BUS_UCLM_2025_V3']
Authorised development-only directed edges: 9
Global source gate passed: True
Decision: SEAL_STAGE11E_R_AUTHORISE_STAGE11F_R_DEVELOPMENT_ONLY_DIRECTED_EDGE_ASSEMBLY_KEEP_STAGE12_DDO2_AND_BLIND_PROHIBITED
Protocol seal: 4362d6a8baed676ee6245971dac40aeb8d2865541df05446ce970ed4431fba72
Embedding manifest hash: a273cdad4892d8b954dad6dfb2c230e7352f68899897466a95852e2c5baa16e2
Axis freeze hash: 98903477c84b7ec6b33258e30f02e8f357ec36e4261c6cbd242fb73ba8c1ec20
Recoverability summary hash: a7ad22dcc25e5bef7763246d79fad64536fcbb87eef4493037712da8f07199be
Stage11F-R handoff hash: 98a285c18d7e680a126b78b5dfa70a4883e7c8cf9536aff2a76528adb42f2a5d
Final record hash: 24d41b7a3a30b1548a81ab1a99d909eb29bf9189fef49709754d1b9cfc8dcca2
Next step: BUILD_STAGE11F_R_DEVELOPMENT_ONLY_DIRE